
**Raw Data Ingestion**
1. import libraries
2. Declare constants for catalog, file path, schemas and table names.
3. Ingest data and add meta data
4. write to tables

In [0]:
from pyspark.sql.functions import current_timestamp, col

In [0]:
# Declare the catalog and schema
CATALOG = "retail_store"
SCHEMA = "bronze"

#Declare table names for sales transaction, customer profile, and product inventory
SALES_TRX = "sales_transactions"
CUST_PROF = "customer_profiles"
PRD_INV = "product_inventory"

# Source File path :- RAW DATA
SRC_DATA_PATH = "/Volumes/retail_store/bronze/raw_data/"

**Ingest _Sales Transaction_ Data**

In [0]:
# Read raw sales transactions data from CSV
df_sales_trx = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")  
    .csv(f"{SRC_DATA_PATH}sales_transaction.csv")
)

# Add ingestion metadata columns: Time ingested and source file.

df_sales_bronze = (
    df_sales_trx
    .withColumn("_processed_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

print(f"Sales transactions raw row count: {df_sales_bronze.count()}")

df_sales_bronze.show(5, truncate=False)

Sales transactions raw row count: 5002
+-------------+----------+---------+-----------------+---------------+-----+--------------------------+----------------------------------------------------------------+
|TransactionID|CustomerID|ProductID|QuantityPurchased|TransactionDate|Price|_processed_timestamp      |_source_file                                                    |
+-------------+----------+---------+-----------------+---------------+-----+--------------------------+----------------------------------------------------------------+
|1            |103       |120      |3                |01/01/23       |30.43|2026-03-16 10:30:05.728664|dbfs:/Volumes/retail_store/bronze/raw_data/sales_transaction.csv|
|2            |436       |126      |1                |01/01/23       |15.19|2026-03-16 10:30:05.728664|dbfs:/Volumes/retail_store/bronze/raw_data/sales_transaction.csv|
|3            |861       |55       |3                |01/01/23       |67.76|2026-03-16 10:30:05.728664|dbfs:/Volumes

**Write to _sales_transactions_ table.**


In [0]:
# Write to Delta — overwrite any previous data
TABLE_PATH = f"{CATALOG}.{SCHEMA}.{SALES_TRX}"
(
    df_sales_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_PATH)
)

print(f"✓ {CATALOG}.{SCHEMA}.{SALES_TRX} written")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {TABLE_PATH}").show()

✓ retail_store.bronze.sales_transactions written
+---------+
|row_count|
+---------+
|     5002|
+---------+



**Ingest _Customer Profiles_ data.**

In [0]:
# Read raw customer profiles data from CSV file
df_customers_profiles = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{SRC_DATA_PATH}customer_profiles.csv")
)
# Add Metadata columns: ingestion time and source file.
df_customers_bronze = (
    df_customers_profiles
    .withColumn("_processed_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

print(f"Customer profiles raw row count: {df_customers_bronze.count()}")
df_customers_bronze.show(5, truncate=False)

Customer profiles raw row count: 1000
+----------+---+------+--------+--------+--------------------------+----------------------------------------------------------------+
|CustomerID|Age|Gender|Location|JoinDate|_processed_timestamp      |_source_file                                                    |
+----------+---+------+--------+--------+--------------------------+----------------------------------------------------------------+
|1         |63 |Other |East    |01/01/20|2026-03-16 10:30:10.957558|dbfs:/Volumes/retail_store/bronze/raw_data/customer_profiles.csv|
|2         |63 |Male  |North   |02/01/20|2026-03-16 10:30:10.957558|dbfs:/Volumes/retail_store/bronze/raw_data/customer_profiles.csv|
|3         |34 |Other |North   |03/01/20|2026-03-16 10:30:10.957558|dbfs:/Volumes/retail_store/bronze/raw_data/customer_profiles.csv|
|4         |19 |Other |NULL    |04/01/20|2026-03-16 10:30:10.957558|dbfs:/Volumes/retail_store/bronze/raw_data/customer_profiles.csv|
|5         |57 |Male  |N

**Write to _customer_profiles_ table.**

In [0]:
# Write to Delta — overwrite any previous data
TABLE_PATH = f"{CATALOG}.{SCHEMA}.{CUST_PROF}"
(
    df_customers_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_PATH)
)

print(f"✓ {CATALOG}.{SCHEMA}.{CUST_PROF} written")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {TABLE_PATH}").show()

✓ retail_store.bronze.customer_profiles written
+---------+
|row_count|
+---------+
|     1000|
+---------+



**Ingest Product Inventory data**

In [0]:
# Read raw Product Inventory data from CSV
df_product_inventory = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{SRC_DATA_PATH}product_inventory.csv")
)
# Add Metadata ingestion date and file info.

df_product_inventory_bronze = (
    df_product_inventory
    .withColumn("_processed_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

print(f"Product Inventory raw row count: {df_product_inventory_bronze.count()}")
df_product_inventory_bronze.show(5, truncate=False)

Product Inventory raw row count: 200
+---------+-----------+---------------+----------+-----+--------------------------+----------------------------------------------------------------+
|ProductID|ProductName|Category       |StockLevel|Price|_processed_timestamp      |_source_file                                                    |
+---------+-----------+---------------+----------+-----+--------------------------+----------------------------------------------------------------+
|1        |Product_1  |Clothing       |22        |46.11|2026-03-16 10:30:16.809078|dbfs:/Volumes/retail_store/bronze/raw_data/product_inventory.csv|
|2        |Product_2  |Home & Kitchen |140       |81.6 |2026-03-16 10:30:16.809078|dbfs:/Volumes/retail_store/bronze/raw_data/product_inventory.csv|
|3        |Product_3  |Home & Kitchen |473       |78.72|2026-03-16 10:30:16.809078|dbfs:/Volumes/retail_store/bronze/raw_data/product_inventory.csv|
|4        |Product_4  |Clothing       |386       |22.06|2026-03-16 10

**Write to _product_inventory_ table**

In [0]:
# Write to Delta — overwrite any previous data
TABLE_PATH = f"{CATALOG}.{SCHEMA}.{PRD_INV}"
(
    df_product_inventory_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_PATH)
)

print(f"✓ {CATALOG}.{SCHEMA}.{PRD_INV} written")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {TABLE_PATH}").show()

✓ retail_store.bronze.product_inventory written
+---------+
|row_count|
+---------+
|      200|
+---------+

